In [3]:
from pyspark.sql import SparkSession
import os

# 1. Tắt Spark Session cũ đang bị lỗi cấu hình
try:
    spark.stop()
    print("Đã tắt Spark Session cũ.")
except:
    pass

# 2. Khởi tạo Spark Session mới với cấu hình CHUẨN MINIO (Giống Thrift Server)
print("⏳ Đang khởi tạo lại Spark Session kết nối MinIO...")

spark = SparkSession.builder \
    .appName("Notebook-Query-Studio") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.my_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.my_catalog.type", "hadoop") \
    .config("spark.sql.catalog.my_catalog.warehouse", "s3a://warehouse/") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.sql.defaultCatalog", "my_catalog") \
    .getOrCreate()
print("✅")

Đã tắt Spark Session cũ.
⏳ Đang khởi tạo lại Spark Session kết nối MinIO...
✅


25/12/05 14:26:58 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [60]:
from pyspark.sql.functions import col, lit, to_date

# Đường dẫn file
file_path = "s3a://warehouse/raw/olist_customers_dataset.csv"

print(f"⏳ Đang kiểm tra file tại: {file_path}")

try:
    # Thử đọc file
    df_raw = spark.read.option("header", "true").csv(file_path)
    
    print("✅ Đã tìm thấy file! Đang xử lý...")
    
    # Chuẩn hóa cột
    df_bronze = df_raw.select(
        col("customer_unique_id").alias("customer_id"),
        col("customer_city").alias("city"),
        col("customer_state").alias("state")
    ).distinct()

    # Thêm cột SCD (Type 1 - Initial Load)
    df_initial = df_bronze \
        .withColumn("start_date", to_date(lit("2023-01-01"))) \
        .withColumn("end_date", lit(None).cast("date")) \
        .withColumn("is_current", lit(True))

    # Ghi vào Iceberg
    print("⏳ Đang ghi vào bảng dim_customers...")
    df_initial.writeTo("my_catalog.default.dim_customers") \
        .using("iceberg") \
        .createOrReplace()

    print(f"🎉 THÀNH CÔNG! Đã nạp {df_initial.count()} dòng khách hàng vào Data Warehouse.")
    df_initial.show(5)

except Exception as e:
    print("\n❌ VẪN LỖI: ")
    print(e)

⏳ Đang kiểm tra file tại: s3a://warehouse/raw/olist_customers_dataset.csv
✅ Đã tìm thấy file! Đang xử lý...
⏳ Đang ghi vào bảng dim_customers...


25/12/03 23:30:49 WARN HadoopTableOperations: Error reading version hint file s3a://warehouse/default/dim_customers/metadata/version-hint.text
java.io.FileNotFoundException: No such file or directory: s3a://warehouse/default/dim_customers/metadata/version-hint.text
	at org.apache.hadoop.fs.s3a.S3AFileSystem.s3GetFileStatus(S3AFileSystem.java:3866)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.innerGetFileStatus(S3AFileSystem.java:3688)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.extractOrFetchSimpleFileStatus(S3AFileSystem.java:5401)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.open(S3AFileSystem.java:1465)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.open(S3AFileSystem.java:1441)
	at org.apache.hadoop.fs.FileSystem.open(FileSystem.java:976)
	at org.apache.iceberg.hadoop.HadoopTableOperations.findVersion(HadoopTableOperations.java:318)
	at org.apache.iceberg.hadoop.HadoopTableOperations.refresh(HadoopTableOperations.java:103)
	at org.apache.iceberg.BaseTransaction.lambda$commitReplaceTransacti

🎉 THÀNH CÔNG! Đã nạp 96219 dòng khách hàng vào Data Warehouse.
+--------------------+---------------+-----+----------+--------+----------+
|         customer_id|           city|state|start_date|end_date|is_current|
+--------------------+---------------+-----+----------+--------+----------+
|9bedaa42178355484...|        chacara|   MG|2023-01-01|    NULL|      true|
|de063bea4f3c9809b...|   pouso alegre|   MG|2023-01-01|    NULL|      true|
|9dc80712fe7b594cb...|coronel joao sa|   BA|2023-01-01|    NULL|      true|
|4fcc54152e34b2fb7...| rio de janeiro|   RJ|2023-01-01|    NULL|      true|
|d22fa88d87b7b3e2e...|       cordeiro|   RJ|2023-01-01|    NULL|      true|
+--------------------+---------------+-----+----------+--------+----------+
only showing top 5 rows



In [22]:
a=df_raw.dropDuplicates(["customer_unique_id"])
a.count()

96096

In [23]:
df_day_2_raw.count()

4592

In [ ]:
# 1. TÍNH TOÁN LOGIC UPDATE 
from pyspark.sql.functions import col, lit, current_date, when

df_current = spark.table("my_catalog.default.dim_customers").filter("is_current = true")
df_day_2_raw = df_current.sample(withReplacement=False, fraction=0.05,seed=2112)
df_day_2_dedup = df_day_2_raw.dropDuplicates(["customer_id"]) 

df_updates_final = df_day_2_dedup.withColumn(
    "city", 
    when(col("city") == "sao paulo", "Ho Chi Minh City")
    .otherwise("Ha Noi")
).withColumn(
    "update_date", current_date()
)

print("⏳ Đang ghi bảng trung gian (Staging)...")
df_updates_final.write \
    .format("iceberg") \
    .mode("overwrite") \
    .saveAsTable("my_catalog.default.staging_updates")

print(f"✅ Đã lưu {df_updates_final.count()} dòng vào bảng Staging.")

⏳ Đang ghi bảng trung gian (Staging)...


25/12/03 23:30:58 WARN HadoopTableOperations: Error reading version hint file s3a://warehouse/default/staging_updates/metadata/version-hint.text
java.io.FileNotFoundException: No such file or directory: s3a://warehouse/default/staging_updates/metadata/version-hint.text
	at org.apache.hadoop.fs.s3a.S3AFileSystem.s3GetFileStatus(S3AFileSystem.java:3866)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.innerGetFileStatus(S3AFileSystem.java:3688)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.extractOrFetchSimpleFileStatus(S3AFileSystem.java:5401)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.open(S3AFileSystem.java:1465)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.open(S3AFileSystem.java:1441)
	at org.apache.hadoop.fs.FileSystem.open(FileSystem.java:976)
	at org.apache.iceberg.hadoop.HadoopTableOperations.findVersion(HadoopTableOperations.java:318)
	at org.apache.iceberg.hadoop.HadoopTableOperations.refresh(HadoopTableOperations.java:103)
	at org.apache.iceberg.BaseTransaction.lambda$commitReplaceTrans

✅ Đã lưu 4886 dòng vào bảng Staging.


In [11]:
df_updates_final.show(5)

+--------------------+----------------+-----+----------+--------+----------+-----------+
|         customer_id|            city|state|start_date|end_date|is_current|update_date|
+--------------------+----------------+-----+----------+--------+----------+-----------+
|008ca52811784a181...|          Ha Noi|   RN|2023-01-01|    NULL|      true| 2025-12-03|
|02bab436d042d111e...|          Ha Noi|   SP|2023-01-01|    NULL|      true| 2025-12-03|
|0651e691ed9e201bf...|          Ha Noi|   SP|2023-01-01|    NULL|      true| 2025-12-03|
|2d6d1699603a346a2...|Ho Chi Minh City|   SP|2023-01-01|    NULL|      true| 2025-12-03|
|3195d6c23c9f61b0b...|          Ha Noi|   SP|2023-01-01|    NULL|      true| 2025-12-03|
+--------------------+----------------+-----+----------+--------+----------+-----------+
only showing top 5 rows



In [62]:
# 2. THỰC HIỆN ONE-SHOT MERGE TỪ STAGING TABLE
spark.sql("""
MERGE INTO my_catalog.default.dim_customers AS target
USING (
    -- UNION TRICK: Lấy dữ liệu từ bảng STAGING (đã cố định)
    SELECT customer_id, city, state, update_date as merge_key, NULL as new_city 
    FROM my_catalog.default.staging_updates
    UNION ALL
    SELECT customer_id, city, state, NULL as merge_key, city as new_city 
    FROM my_catalog.default.staging_updates
) AS source
ON target.customer_id = source.customer_id 
   AND target.is_current = true 
   AND source.merge_key IS NOT NULL

WHEN MATCHED THEN 
  UPDATE SET target.is_current = false, target.end_date = source.merge_key

WHEN NOT MATCHED AND source.merge_key IS NULL THEN 
  INSERT (customer_id, city, state, start_date, end_date, is_current)
  VALUES (source.customer_id, source.new_city, source.state, current_date(), NULL, true)
""")

print("✅ MERGE THÀNH CÔNG")

[Stage 30:>                                                         (0 + 5) / 5]

✅ MERGE THÀNH CÔNG! (Chắc chắn 100%)


In [63]:
# Lấy 1 ID từ bảng STAGING (Bảng trung gian em vừa tạo)
# Thay 'source_updates' bằng 'my_catalog.default.staging_updates'
sample_id = spark.sql("SELECT customer_id FROM my_catalog.default.staging_updates LIMIT 1").collect()[0][0]

print(f"🔎 Soi lịch sử khách hàng: {sample_id}")

# Kiểm tra trong bảng GỐC xem đã cập nhật chưa
spark.sql(f"""
    SELECT customer_id, city, start_date, end_date, is_current 
    FROM my_catalog.default.dim_customers 
    WHERE customer_id = '{sample_id}'
    ORDER BY start_date
""").show(truncate=False)

🔎 Soi lịch sử khách hàng: 000c8bdb58a29e7115cfc257230fb21b
+--------------------------------+--------------+----------+----------+----------+
|customer_id                     |city          |start_date|end_date  |is_current|
+--------------------------------+--------------+----------+----------+----------+
|000c8bdb58a29e7115cfc257230fb21b|belo horizonte|2023-01-01|2025-12-03|false     |
|000c8bdb58a29e7115cfc257230fb21b|Ha Noi        |2025-12-03|NULL      |true      |
+--------------------------------+--------------+----------+----------+----------+



In [64]:
spark.sql("DROP TABLE IF EXISTS my_catalog.default.staging_updates")
print("🧹 Đã dọn dẹp bảng tạm.")

🧹 Đã dọn dẹp bảng tạm.


In [65]:
from pyspark.sql.functions import col

print("⏳ Lịch sử các thay đổi (Snapshots) của bảng:")

# Đổi .history thành .snapshots
df_history = spark.sql("""
    SELECT committed_at, snapshot_id, operation, summary 
    FROM my_catalog.default.dim_customers.snapshots 
    ORDER BY committed_at
""")

df_history.show(truncate=False)

⏳ Lịch sử các thay đổi (Snapshots) của bảng:
+-----------------------+-------------------+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |operation|summary                                                                                                                                                                                                                                                                                         

In [68]:
old_snapshot_id = 2763217490268994471 

print(f"🔙 Quay ngược thời gian về Snapshot: {old_snapshot_id}")

df_past = spark.read \
    .option("snapshot-id", old_snapshot_id) \
    .table("my_catalog.default.dim_customers")

# Kiểm tra lại khách hàng (Lúc này họ CHƯA chuyển nhà)
target_id = sample_id 

print(f"🔎 Dữ liệu tại quá khứ của khách hàng: {target_id}")
df_past.filter(f"customer_id = '{target_id}'").show(truncate=False)

🔙 Quay ngược thời gian về Snapshot: 2763217490268994471
🔎 Dữ liệu tại quá khứ của khách hàng: 000c8bdb58a29e7115cfc257230fb21b
+--------------------------------+--------------+-----+----------+--------+----------+
|customer_id                     |city          |state|start_date|end_date|is_current|
+--------------------------------+--------------+-----+----------+--------+----------+
|000c8bdb58a29e7115cfc257230fb21b|belo horizonte|MG   |2023-01-01|NULL    |true      |
+--------------------------------+--------------+-----+----------+--------+----------+



In [69]:
from pyspark.sql.functions import col, struct, collect_list

# 1. Đọc dữ liệu từ MinIO
print("⏳ Đang đọc dữ liệu Orders và Items...")
df_orders = spark.read.option("header", "true").csv("s3a://warehouse/raw/olist_orders_dataset.csv")
df_items = spark.read.option("header", "true").csv("s3a://warehouse/raw/olist_order_items_dataset.csv")

# 2. Xử lý bảng Items: Gom thông tin chi tiết thành STRUCT
# Thay vì lưu rời rạc, ta đóng gói (product_id, price, freight) thành 1 cục gọi là 'item_detail'
df_items_struct = df_items.select(
    "order_id", 
    struct(
        col("product_id"), 
        col("price").cast("float"), 
        col("freight_value").cast("float")
    ).alias("item_detail")
)

# 3. Gom nhóm (Aggregation): Biến nhiều dòng thành 1 dòng chứa ARRAY
# Group theo order_id, gom tất cả item_detail vào một list
df_items_nested = df_items_struct.groupBy("order_id").agg(
    collect_list("item_detail").alias("line_items")
)

# 4. Join ngược lại với bảng Orders để ra bảng Fact cuối cùng
# Đây là bảng "Denormalized" - Tối ưu cho đọc (Read Optimized)
df_fact = df_orders.join(df_items_nested, "order_id", "left")

print("⏳ Đang ghi bảng Fact Nested vào Iceberg...")
df_fact.writeTo("my_catalog.default.fact_orders_nested") \
    .using("iceberg") \
    .createOrReplace()

print("✅ HOÀN TẤT DATA MODELING (STRUCT/ARRAY)!")

⏳ Đang đọc dữ liệu Orders và Items...
⏳ Đang ghi bảng Fact Nested vào Iceberg...


✅ HOÀN TẤT DATA MODELING (STRUCT/ARRAY)!


25/12/03 23:33:38 WARN HadoopTableOperations: Error reading version hint file s3a://warehouse/default/fact_orders_nested/metadata/version-hint.text
java.io.FileNotFoundException: No such file or directory: s3a://warehouse/default/fact_orders_nested/metadata/version-hint.text
	at org.apache.hadoop.fs.s3a.S3AFileSystem.s3GetFileStatus(S3AFileSystem.java:3866)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.innerGetFileStatus(S3AFileSystem.java:3688)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.extractOrFetchSimpleFileStatus(S3AFileSystem.java:5401)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.open(S3AFileSystem.java:1465)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.open(S3AFileSystem.java:1441)
	at org.apache.hadoop.fs.FileSystem.open(FileSystem.java:976)
	at org.apache.iceberg.hadoop.HadoopTableOperations.findVersion(HadoopTableOperations.java:318)
	at org.apache.iceberg.hadoop.HadoopTableOperations.refresh(HadoopTableOperations.java:103)
	at org.apache.iceberg.BaseTransaction.lambda$commitReplac

In [70]:
# Load lại bảng vừa tạo
df_final = spark.table("my_catalog.default.fact_orders_nested")

print("--- SCHEMA CỦA BẢNG HIỆN ĐẠI (NESTED) ---")
df_final.printSchema()

print("\n--- DỮ LIỆU MẪU (1 Đơn hàng chứa nhiều sản phẩm) ---")
# Lọc tìm đơn hàng nào có nhiều hơn 1 sản phẩm để demo cho rõ
from pyspark.sql.functions import size
df_final.filter(size(col("line_items")) > 1) \
    .select("order_id", "order_status", "line_items") \
    .show(1, truncate=False)

--- SCHEMA CỦA BẢNG HIỆN ĐẠI (NESTED) ---
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)
 |-- line_items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- product_id: string (nullable = true)
 |    |    |-- col2: float (nullable = true)
 |    |    |-- col3: float (nullable = true)


--- DỮ LIỆU MẪU (1 Đơn hàng chứa nhiều sản phẩm) ---
+--------------------------------+------------+--------------------------------------------------------------------------------------------------+
|order_id                        |order_status|line_items                                                 

In [4]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd

# 1. CẤU HÌNH HIỂN THỊ
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 100)

# 2. GIAO DIỆN (UI)
header = widgets.HTML(
    value="<div style='background-color: #2E86C1; color: white; padding: 12px; border-radius: 8px; margin-bottom: 10px;'>"
          "<h2 style='margin: 0;'>🚀 DATA LAKEHOUSE QUERY STUDIO</h2>"
          "<p style='margin: 5px 0 0 0;'>Công cụ truy vấn hệ thống Spark - Iceberg - MinIO</p>"
          "</div>"
)

# Menu chọn kịch bản
dropdown = widgets.Dropdown(
    options=[
        ('--- Chọn kịch bản Demo ---', ''),
        ('1. [HẠ TẦNG] Kiểm tra danh sách bảng (Show Tables)', 'SHOW TABLES IN my_catalog.default'),
        ('2. [SCD TYPE 2] Soi lịch sử thay đổi Khách Hàng', 
         'SELECT customer_id, city, is_current, start_date, end_date FROM my_catalog.default.dim_customers WHERE customer_id IN (SELECT customer_id FROM my_catalog.default.dim_customers WHERE is_current=false LIMIT 1) ORDER BY customer_id, start_date'),
        ('3. [NESTED DATA] Soi cấu trúc Đơn Hàng (Array/Struct)', 
         'SELECT order_id, customer_id, line_items FROM my_catalog.default.fact_orders_nested WHERE size(line_items) > 1 LIMIT 3'),
        ('4. [THỐNG KÊ] Top 5 thành phố đông khách nhất', 
         'SELECT city, count(*) as cnt FROM my_catalog.default.dim_customers WHERE is_current=true GROUP BY city ORDER BY cnt DESC LIMIT 5')
    ],
    value='',
    description='<b>Kịch bản:</b>',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='80%')
)

# Ô NHẬP SQL (Cái này lúc nãy bị thiếu)
sql_area = widgets.Textarea(
    value='',
    placeholder='Câu lệnh SQL sẽ hiện ở đây khi em chọn kịch bản. Hoặc em có thể tự gõ lệnh...',
    description='<b>SQL Code:</b>',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='98%', height='100px')
)

# Nút chạy
btn_run = widgets.Button(
    description=' CHẠY TRUY VẤN',
    button_style='primary', # Màu xanh dương
    icon='play',
    layout=widgets.Layout(width='200px', height='40px', margin='10px 0px')
)

# Vùng hiển thị kết quả
out = widgets.Output(layout={'border': '1px solid #ddd', 'padding': '15px', 'margin-top': '10px'})

# 3. XỬ LÝ (BACKEND)
def on_dropdown_change(change):
    # Khi chọn menu thì tự động điền code vào ô SQL
    if change['new']:
        sql_area.value = change['new']

def on_click(b):
    with out:
        clear_output()
        # Lấy code từ ô nhập liệu (sql_area) thay vì dropdown
        sql = sql_area.value.strip()
        
        if not sql:
            print("⚠️ Vui lòng chọn kịch bản hoặc nhập code SQL!")
            return

        print(f"⏳ Đang chạy lệnh: {sql} ...")
        try:
            df = spark.sql(sql).toPandas()
            print(f"✅ Tìm thấy {len(df)} dòng dữ liệu.\n")
            if not df.empty:
                display(df.style.set_table_styles([
                    {'selector': 'th', 'props': [('background-color', '#2E86C1'), ('color', 'white'), ('text-align', 'center')]},
                    {'selector': 'td', 'props': [('border', '1px solid #ddd'), ('padding', '8px')]},
                    {'selector': 'tr:nth-of-type(even)', 'props': [('background-color', '#f2f2f2')]}
                ]))
            else:
                print("⚠️ Bảng rỗng.")
        except Exception as e:
            print(f"❌ LỖI: {e}")

# Gắn sự kiện
dropdown.observe(on_dropdown_change, names='value')
btn_run.on_click(on_click)

# 4. HIỂN THỊ (Đã thêm sql_area vào)
display(header, dropdown, sql_area, btn_run, out)

HTML(value="<div style='background-color: #2E86C1; color: white; padding: 12px; border-radius: 8px; margin-bot…

Dropdown(description='<b>Kịch bản:</b>', layout=Layout(width='80%'), options=(('--- Chọn kịch bản Demo ---', '…

Textarea(value='', description='<b>SQL Code:</b>', layout=Layout(height='100px', width='98%'), placeholder='Câ…

Button(button_style='primary', description=' CHẠY TRUY VẤN', icon='play', layout=Layout(height='40px', margin=…

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…